# Q02 — DataFrame Construction, Ascending Sort, and Sales-Derived Columns

**Python for Data Science — Company-Reported Interview Questions**

| Metadata | Details |
|---|---|
| **Company** | CVS Health |
| **Position** | Data Scientist |
| **Primary Category** | 01 — Pandas Fundamentals and Data Structures |
| **Secondary Topics** | DataFrame Construction; Sorting; Column Assignment; Vectorized Arithmetic; Aggregation |
| **Difficulty** | Intermediate (reported task) → Advanced/Expert (educational extensions) |
| **Source Type** | Candidate-reported interview task; partially specified |
| **Interview Year** | Not independently verified |
| **Original Source** | [Glassdoor — CVS Health Data Scientist Interview](https://www.glassdoor.com/Interview/1st-round-Live-coding-questions-SQL-There-were-3-tables-and-the-question-was-what-percentage-of-2020-paid-amt-was-each-QTN_7022919.htm) |

> **Authenticity boundary:** The candidate reported creating a Pandas DataFrame, sorting ascending, and adding two columns associated with total sales. The exact input table, two column definitions, and expected outputs were **not published**. This notebook presents an **adapted, reproducible exercise**, not an exact transcript of the original question.

## 1. Reported Interview Task

**Candidate report (paraphrased):** Create a Pandas DataFrame, sort its observations in ascending order, and add two new columns related to total sales.

### 1.1 Adapted, fully specified coding question

You are given six product-level sales records containing `order_id`, `product`, and `sales`.

1. Construct a Pandas DataFrame with the specified schema.
2. Sort rows by `sales` in **ascending** order. For ties, apply `order_id` as a secondary ascending key to make the result deterministic.
3. Add exactly two derived columns:
   - `total_sales`: the sum of `sales` across **all** rows, repeated on every row;
   - `sales_share_pct`: each row's percentage contribution to total sales.
4. Return the columns in the order `order_id`, `product`, `sales`, `total_sales`, `sales_share_pct`.
5. Preserve `order_id` as an integer column, `sales` as floating-point currency values, and use a fresh `RangeIndex` after sorting.

**Assumptions for the core exercise:** `sales` contains finite, nonnegative values and the aggregate sales total is strictly positive. Additional edge cases are considered later.


## 2. Mathematical Definition

For observations $i = 1, \ldots, n$ with sales values $s_i$, define

$$
T = \sum_{i=1}^{n} s_i, \qquad T > 0.
$$

The two derived columns are

$$
\mathrm{total\_sales}_i = T,
$$

$$
\mathrm{sales\_share\_pct}_i = 100 \frac{s_i}{T}.
$$

The order of observations changes after sorting, but **the total sales and each order's percentage contribution do not**.

When all sales are finite and nonnegative,

$$
\sum_{i=1}^{n} \mathrm{sales\_share\_pct}_i = 100
$$

up to floating-point rounding.


## 3. Data Construction and Schema Verification

The input below is **synthetic**, introduced solely to make the candidate-reported task executable.

In [1]:
import numpy as np
import pandas as pd

records = [
    {"order_id": 101, "product": "Laptop",   "sales": 1200.0},
    {"order_id": 102, "product": "Monitor",  "sales":  300.0},
    {"order_id": 103, "product": "Keyboard", "sales":  100.0},
    {"order_id": 104, "product": "Mouse",    "sales":   50.0},
    {"order_id": 105, "product": "Tablet",   "sales":  300.0},
    {"order_id": 106, "product": "Headset",  "sales":   50.0},
]

orders = pd.DataFrame.from_records(records)
print(orders.to_string(index=False))
print("\nSchema:")
print(orders.dtypes)

 order_id  product  sales
      101   Laptop 1200.0
      102  Monitor  300.0
      103 Keyboard  100.0
      104    Mouse   50.0
      105   Tablet  300.0
      106  Headset   50.0

Schema:
order_id      int64
product      object
sales       float64
dtype: object


## 4. Standard Pandas Solution

Sorting by two columns specifies a deterministic order even when two products have identical sales. The operation is followed by `reset_index(drop=True)` to give the sorted DataFrame a new positional index.

`assign` adds the derived columns without needing a row-wise Python loop. `DataFrame.assign` returns a new DataFrame; its second expression is evaluated against the DataFrame with `total_sales` already defined.

In [2]:
def build_sales_report(orders: pd.DataFrame) -> pd.DataFrame:
    required = {"order_id", "product", "sales"}
    if not required.issubset(orders.columns):
        raise ValueError(f"Missing required columns: {sorted(required - set(orders.columns))}")
    if orders.empty:
        raise ValueError("At least one sales record is required")

    sales = orders["sales"]
    if not pd.api.types.is_numeric_dtype(sales):
        raise TypeError("sales must have a numeric dtype")
    if sales.isna().any() or not np.isfinite(sales.to_numpy(dtype=float)).all():
        raise ValueError("sales values must be finite and non-missing")
    if (sales < 0).any():
        raise ValueError("sales must be nonnegative")

    total = float(sales.sum())
    if not np.isfinite(total) or total <= 0:
        raise ValueError("Total sales must be finite and positive")

    ordered = orders.loc[:, ["order_id", "product", "sales"]].sort_values(
        by=["sales", "order_id"], ascending=[True, True]
    ).reset_index(drop=True)

    return ordered.assign(
        total_sales=total,
        sales_share_pct=lambda frame: 100.0 * frame["sales"] / frame["total_sales"],
    )

report = build_sales_report(orders)
print(report.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

 order_id  product   sales  total_sales  sales_share_pct
      104    Mouse   50.00      2000.00             2.50
      106  Headset   50.00      2000.00             2.50
      103 Keyboard  100.00      2000.00             5.00
      102  Monitor  300.00      2000.00            15.00
      105   Tablet  300.00      2000.00            15.00
      101   Laptop 1200.00      2000.00            60.00


## 5. Expected Results

| order_id | product | sales | total_sales | sales_share_pct |
|---:|---|---:|---:|---:|
| 104 | Mouse | 50.00 | 2000.00 | 2.50 |
| 106 | Headset | 50.00 | 2000.00 | 2.50 |
| 103 | Keyboard | 100.00 | 2000.00 | 5.00 |
| 102 | Monitor | 300.00 | 2000.00 | 15.00 |
| 105 | Tablet | 300.00 | 2000.00 | 15.00 |
| 101 | Laptop | 1200.00 | 2000.00 | 60.00 |

**Check:** \(50+50+100+300+300+1200=2000\), and \(2.5+2.5+5+15+15+60=100\%\).

## 6. Automated Verification

Assertions test not only the visible table but also ordering, schema, index, numerical accuracy, and preservation of the source data.

In [3]:
assert report["order_id"].tolist() == [104, 106, 103, 102, 105, 101]
assert report.columns.tolist() == [
    "order_id", "product", "sales", "total_sales", "sales_share_pct"
]
assert report.index.equals(pd.RangeIndex(len(report)))
assert report["order_id"].dtype.kind in "iu"
np.testing.assert_allclose(report["total_sales"], 2000.0)
np.testing.assert_allclose(
    report["sales_share_pct"].to_numpy(),
    np.array([2.5, 2.5, 5.0, 15.0, 15.0, 60.0]),
)
assert np.isclose(report["sales_share_pct"].sum(), 100.0)
assert "total_sales" not in orders.columns  # original unchanged
print("All Q02 core tests passed.")

All Q02 core tests passed.


## 7. Conceptual Analysis

### 7.1 DataFrame construction

`DataFrame.from_records` converts a sequence of mapping-like records into a labeled, columnar table. For real interview questions, explicit verification of column names and inferred dtypes helps prevent silent errors.

### 7.2 Sorting and indices

`sort_values` reorders rows by column values. Unless the index is reset, the **original row labels** are retained, which may confuse later `.loc`-based selection or comparisons against another DataFrame.

A secondary sort key resolves ties. This is important for reproducibility, although stable sorting details depend on how the operation and sorting keys are specified.

### 7.3 Scalar broadcasting and vectorized operations

`total_sales` is a scalar broadcast to every row. Percentage contributions are calculated by dividing the entire `sales` Series by a scalar, avoiding explicit row-wise iteration.

### 7.4 Column dtypes

Numeric currency values are shown as floats for convenience. For exact monetary accounting, decimal or integer-minor-unit representations may be preferable to binary floating-point arithmetic.

### 7.5 Index semantics

This example uses a `RangeIndex` after sorting. The `order_id` column is a separate business identifier and must not be confused with the DataFrame index.

## 8. Computational Complexity

Let \(n\) denote the number of sales records and assume a fixed number of columns.

| Step | Typical time | Additional memory |
|---|---|---|
| Build DataFrame from records | \(O(n)\) | \(O(n)\) |
| Validate values | \(O(n)\) | \(O(n)\) worst case for masks/conversion |
| Compute sales total | \(O(n)\) | \(O(1)\) apart from implementation buffers |
| Sort records | \(O(n\log n)\) | implementation-dependent, typically \(O(n)\) |
| Add repeated total column | \(O(n)\) | \(O(n)\) |
| Compute percentage column | \(O(n)\) | \(O(n)\) |

The full procedure is dominated by sorting:

\[
T(n)=O(n\log n), \qquad M(n)=O(n)
\]

These are **typical algorithmic upper-bound models**, not universal guarantees for arbitrary Pandas index types or object-dtype sorting. If the output need not be sorted, the transformation can be implemented in approximately \(O(n)\) time.

## 9. Alternative Implementation — Explicit Assignment

The same exercise can be implemented without `assign`, using explicit column assignment on a copied and sorted DataFrame.

In [4]:
alternative = (
    orders[["order_id", "product", "sales"]]
    .sort_values(["sales", "order_id"])
    .reset_index(drop=True)
    .copy()
)
alternative["total_sales"] = alternative["sales"].sum()
alternative["sales_share_pct"] = (
    alternative["sales"] / alternative["total_sales"] * 100.0
)

pd.testing.assert_frame_equal(report, alternative)
print("Alternative implementation matches the primary solution.")

Alternative implementation matches the primary solution.


## 10. Edge Cases and Robustness

Advanced candidates should identify input conditions that were not covered in the brief original report:

1. **Empty input:** no meaningful positive total exists; this implementation raises `ValueError`.
2. **Zero total sales:** percentages would be undefined; this implementation rejects the input.
3. **Null/NaN/non-finite amounts:** reject or define an explicit business policy before aggregation.
4. **Negative sales:** returns or refunds may legitimately exist, but require a different task definition.
5. **Duplicate identifiers:** determine whether duplicate `order_id` values are allowed and whether deduplication is required.
6. **Equal sales amounts:** use a secondary key to provide deterministic output.
7. **Floating-point money:** consider an exact representation for financial reporting.

The synthetic example and its validation policy are **educational additions**, not verified CVS interview requirements.

In [5]:
# Check deliberately invalid cases.
invalid_cases = {
    "empty": orders.iloc[0:0],
    "zero_total": pd.DataFrame({"order_id": [1], "product": ["X"], "sales": [0.0]}),
    "missing_sale": pd.DataFrame({"order_id": [1], "product": ["X"], "sales": [np.nan]}),
    "negative_sale": pd.DataFrame({"order_id": [1], "product": ["X"], "sales": [-1.0]}),
}
for label, bad_input in invalid_cases.items():
    try:
        build_sales_report(bad_input)
    except (ValueError, TypeError) as exc:
        print(f"{label}: rejected ({type(exc).__name__})")
    else:
        raise AssertionError(f"Expected {label} to fail")

empty: rejected (ValueError)
zero_total: rejected (ValueError)
missing_sale: rejected (ValueError)
negative_sale: rejected (ValueError)


## 11. Advanced Follow-up Questions (Original Educational Extensions)

**Q1. Why is sorting the most expensive step?**  
For general comparison-based ordering, sorting \(n\) rows typically takes \(O(n\log n)\) comparisons, whereas summation and vectorized column arithmetic each traverse the data in \(O(n)\).

**Q2. What is the difference between `sort_values` and `sort_index`?**  
The former sorts by data values; the latter sorts axis labels. They are not interchangeable.

**Q3. Why should a total be computed across the complete dataset rather than after arbitrary filtering?**  
The denominator defines the population for the percentage. Filtering before summation changes the interpretation of the contribution shares.

**Q4. When might `groupby(...).transform('sum')` be appropriate instead?**  
When each row's contribution must be measured against a **group-level** total (e.g., revenue within its region), not the grand total across the DataFrame.

**Q5. Does adding a scalar to a column imply repeated Python execution per row?**  
No. Pandas can broadcast scalar values and perform array-oriented operations without an explicit Python-level row loop.

**Q6. How would you handle exact decimal money?**  
Store monetary values as integer cents (when scale is fixed) or use a decimal representation and carefully manage rounding. Floating-point display formatting alone does not make calculations exact.

## 12. Analytical Interpretation

The resulting `sales_share_pct` column is a **composition measure**: it describes the relative contribution of each record to total observed sales. It is not a growth rate, profitability measure, or prediction.

Sorting makes the smallest contributions easiest to inspect first. In real analytical reporting, use a clearly defined population, treatment of returns, currency units, and a consistent aggregation level before drawing business conclusions.

A strong interview explanation connects DataFrame construction, schema integrity, index handling, deterministic sorting, vectorized computation, and validation with the statistical meaning of the output.

## 13. Interview-Ready Summary

**Concise answer:** Build the DataFrame from the records, check its schema, calculate total sales with `sum`, use `sort_values` with an explicit ascending order, and compute two columns with vectorized Pandas arithmetic. Reset the index if a clean sequential index is required, and validate both ordering and the sum of percentage contributions.

**Authenticity reminder:** The original public CVS Health interview report establishes the broad task but **does not establish this synthetic dataset, the column definitions, or the additional follow-up questions**.

**Source:** [CVS Health Data Scientist candidate report — Glassdoor](https://www.glassdoor.com/Interview/1st-round-Live-coding-questions-SQL-There-were-3-tables-and-the-question-was-what-percentage-of-2020-paid-amt-was-each-QTN_7022919.htm)